# Embedding, vector store and search (KAN-97)
Gözde Nur Özdemir · ELIZA · Sprint 2

**What the course asks for in Sprint 2:** *"the most basic vector database and retrieval over your documents"*, with the parameters in `config/rag.yaml`, and by 11/11: *"retrieval returns passages for a handful of test questions"*.

**What this notebook does:** it turns our 105 chunks into vectors, stores them in a vector database, and returns the 5 best chunks for a question. It first checks the embedding model chosen in Sprint 1 on my questions. The other settings (words BM25 ignores, search mode) are **chosen and tested in `src/evaluation.ipynb`** and written in `config/rag.yaml`; this notebook only builds the system with them.

| Step | What | Setting from `config/rag.yaml` |
|---|---|---|
| 1 | Load the settings and the chunks | |
| 2 | Was the Sprint 1 embedding model a good choice? Checked with Recall@5, MRR and Precision@5 | `embedding.model`, D10 |
| 3 | Turn the chunks into vectors (embedding) | `embedding.model` |
| 4 | Store the vectors in Chroma | `vector_store` |
| 5 | Keyword index (BM25) | `bm25.jury_words` |
| 6 | Search: dense (E1) and hybrid (E2) | `retrieval` |
| 7 | My questions, one by one (11/11 check) | |
| 8 | A picture of the chunks | |

**Questions used:** only my questions on Annex III 3(b): B-01 to B-06 for the scores, and the safety questions B-S1 to B-S3 in step 7.

**Install once, in the terminal** (not in the notebook), in the same Python environment as Jupyter:
```
pip install sentence-transformers chromadb rank_bm25 pandas pyyaml requests scikit-learn umap-learn matplotlib
```

## 1. Load the settings and the chunks
The chunks come from `src/chunking.ipynb`. The notebook works in the repository (opened from `src/`) or in one folder that holds the notebook, `rag.yaml`, `chunks.jsonl` and `eval-set-v1.csv`.

In [ ]:
import csv, json, os, re
from collections import Counter
import numpy as np, pandas as pd, requests, yaml
pd.set_option("display.max_colwidth", None)
np.random.seed(42)

IN_REPO = os.path.exists("../config/rag.yaml")      # opened from src/ in the repository, or from one folder with all files
P = lambda path: path if IN_REPO else os.path.basename(path)
CONFIG = yaml.safe_load(open(P("../config/rag.yaml")))
chunks = [json.loads(line) for line in open(P("../data/chunks.jsonl"), encoding="utf-8")]
questions = {r["id"]: r for r in csv.DictReader(open(P("../docs/eval/eval-set-v1.csv"), encoding="utf-8-sig"))}
ref_of = {c["chunk_id"]: c["ref"] for c in chunks}
words = lambda text: re.findall(r"[a-z0-9]+", text.lower())
print(len(chunks), "chunks,", len(questions), "questions")
CONFIG

## 2. Was the Sprint 1 embedding model a good choice?
An embedding model turns a text into a list of numbers (a vector); texts with a similar meaning get similar numbers, even when they share no words. A question like *"our AI grades exams"* can then find Annex III 3(b) (*"evaluate learning outcomes"*).

In Sprint 1 the design document (decision D10) compared three open models that run on a CPU and chose **`bge-base-en-v1.5`**, from published results, and said it would be checked on our questions. We check it here: each model embeds the 105 chunks and searches my questions **B-01 to B-06**. Scores, as in the instructor's slides (*"compute Recall@k / MRR on gold IDs"*):
- **Recall@5:** share of the gold provisions in the top 5 (4 gold provisions, 2 found = 0.5)
- **MRR:** how high the first correct chunk is (1.0 = always first, 0.5 = second)
- **Precision@5:** share of the top 5 that are gold provisions (unrelated chunks lower it)

| Model | Numbers per text | In D10 |
|---|---|---|
| `sentence-transformers/all-MiniLM-L6-v2` | 384 | very small, weaker on retrieval |
| `BAAI/bge-small-en-v1.5` | 384 | small, proposed in Spike 01 |
| `BAAI/bge-base-en-v1.5` | 768 | **chosen in Sprint 1**: stronger on retrieval, MIT licence |

**Rule:** the Sprint 1 choice stays unless another model is better on **both** Recall@5 and MRR.

In [ ]:
dev = [questions[q] for q in ["B-01", "B-02", "B-03", "B-04", "B-05", "B-06"]]

def score(search, k=5):
    """Recall@k, MRR and Precision@k over my questions B-01 to B-06."""
    recall, rr, precision = [], [], []
    for q in dev:
        gold = [g.strip() for g in q["gold_source"].split(";")]
        refs = [ref_of[c] for c in search(q["question"])[:k]]
        hit = lambda r: any(r.startswith(g) or g.startswith(r) for g in gold)
        recall.append(np.mean([any(r.startswith(g) or g.startswith(r) for r in refs) for g in gold]))
        rr.append(next((1 / n for n, r in enumerate(refs, 1) if hit(r)), 0))
        precision.append(np.mean([hit(r) for r in refs]))
    return {"Recall@5" if k == 5 else f"Recall@{k}": round(np.mean(recall), 2), "MRR": round(np.mean(rr), 2),
            "Precision@5" if k == 5 else f"Precision@{k}": round(np.mean(precision), 2)}

import time
from sentence_transformers import SentenceTransformer

def dense_search(model, vectors, instruction):
    """A search that ranks the chunks by cosine similarity to the question."""
    return lambda q: [chunks[n]["chunk_id"] for n in np.argsort(-(vectors @ model.encode([instruction + q], normalize_embeddings=True)[0]))]

models, rows = {}, []
for name in CONFIG["embedding"]["candidates"]:
    model = SentenceTransformer(name, device="cpu")
    start = time.time()
    vectors = model.encode([c["embed_text"] for c in chunks], normalize_embeddings=True)
    seconds = round(time.time() - start, 1)
    instruction = CONFIG["embedding"]["query_instruction"] if "bge" in name else ""
    models[name] = (model, vectors, instruction)
    rows.append({"model": name, "numbers per text": vectors.shape[1], **score(dense_search(*models[name])), "seconds for 105 chunks": seconds})

model_test = pd.DataFrame(rows).sort_values(["Recall@5", "MRR"], ascending=False)
current = model_test.set_index("model").loc[CONFIG["embedding"]["model"]]          # the Sprint 1 choice (D10)
clearly_better = model_test[(model_test["Recall@5"] > current["Recall@5"]) & (model_test["MRR"] > current["MRR"])]
EMBEDDING = clearly_better.iloc[0]["model"] if len(clearly_better) else CONFIG["embedding"]["model"]
model, vectors, INSTRUCTION = models[EMBEDDING]
print("Kept:" if EMBEDDING == CONFIG["embedding"]["model"] else "Changed to:", EMBEDDING)
model_test

**Comment:** No model is better on both scores: bge-small found a few more gold provisions (Recall@5 0.48 against 0.40), but bge-base put the first correct chunk much higher (MRR 0.53 against 0.26), and MiniLM was lower on both. So the Sprint 1 choice, bge-base, stays; with six questions the difference is small, and KAN-99 compares them again on the val split.

## 3. Turn the chunks into vectors (embedding)
We embed the full `embed_text` of each chunk (heading + the sentence before a point + the provision), with nothing removed, with the model kept in step 2. As its model card recommends, questions get a short instruction in front; chunks do not.

In [ ]:
model, vectors, INSTRUCTION = models[EMBEDDING]
embed = lambda texts: model.encode(texts, normalize_embeddings=True)

print(EMBEDDING, "->", vectors.shape, "(105 chunks x numbers per chunk)")

i = {c["chunk_id"]: n for n, c in enumerate(chunks)}
print("ART6-3-a vs ART6-3-b (two points of the same rule):", round(float(vectors[i["ART6-3-a"]] @ vectors[i["ART6-3-b"]]), 2))
print("ART6-3-a vs ART113-3-c-i (a different topic):      ", round(float(vectors[i["ART6-3-a"]] @ vectors[i["ART113-3-c-i"]]), 2))

**Comment:** _(write 1–2 sentences after running)_

## 4. Store them in Chroma
Chroma (D11) keeps the vectors, the texts and the metadata, and returns the closest chunks with cosine distance. It is saved in `data/vector-store/`; it is not in Git, because this notebook rebuilds it in seconds.

In [ ]:
import chromadb
store = chromadb.PersistentClient(path=P(CONFIG["vector_store"]["path"]))
if "chunks" in [c.name for c in store.list_collections()]:
    store.delete_collection("chunks")
col = store.create_collection("chunks", metadata={"hnsw:space": "cosine"})
col.add(ids=[c["chunk_id"] for c in chunks], embeddings=vectors.tolist(), documents=[c["text"] for c in chunks],
        metadatas=[{"ref": c["ref"], "version_date": c["version_date"]} for c in chunks])
print(col.count(), "chunks stored")

## 5. Keyword index (BM25)
BM25 (D12) gives a high score to chunks that share the question's words, especially rare ones. It finds exact terms such as "Annex III" or "Article 113" that the embedding can miss. It reads the text in lower case and ignores the 11 words chosen in `evaluation.ipynb` (the LLM jury's words, for example "the", "of", "shall"); "not", "no" and "nor" are kept, because they turn a rule around.

In [ ]:
from rank_bm25 import BM25Okapi
IGNORE = set(CONFIG["bm25"]["jury_words"]) if CONFIG["bm25"]["remove"] == "jury" else set()
keep = lambda text: [w for w in words(text) if w not in IGNORE]
bm25 = BM25Okapi([keep(c["embed_text"]) for c in chunks])
print("BM25 index over", len(chunks), "chunks; ignored words:", sorted(IGNORE))

## 6. Search: dense (E1) and hybrid (E2)
- **Dense (E1, the baseline):** the closest vectors in Chroma.
- **Hybrid (E2):** the dense list and the BM25 list joined with Reciprocal Rank Fusion: each chunk gets `1 / (60 + rank)` from each list, and the two are added. 60 is the value from the RRF paper (Cormack et al., 2009).

Both return the top 5 (`top_k`). On my questions hybrid search was better (`evaluation.ipynb`), so it is the setting in `config/rag.yaml`.

In [ ]:
R = CONFIG["retrieval"]

def dense(question):
    q = embed([INSTRUCTION + question])[0]
    return col.query(query_embeddings=[q.tolist()], n_results=R["candidates"])["ids"][0]

def keyword(question):
    return [chunks[n]["chunk_id"] for n in np.argsort(-bm25.get_scores(keep(question)))[:R["candidates"]]]

def hybrid(question):
    rrf = Counter()
    for ranked in (dense(question), keyword(question)):
        for rank, cid in enumerate(ranked, 1):
            rrf[cid] += 1 / (R["rrf_k"] + rank)
    return [cid for cid, _ in rrf.most_common()]          # full ranking; the top_k are taken by the caller

[ref_of[c] for c in hybrid("Is an AI tool that grades students' essays high-risk?")[:R["top_k"]]]

**Comment:** _(write 1–2 sentences after running)_

## 7. My questions, one by one (11/11 check)
For each of my questions: the top 5 chunks of each search, and the **gold source**; ✓ = a gold provision. B-01 to B-06 should find their gold provisions. B-S1 to B-S3 are safety questions (refuse, out of scope, no answer in the law): the search still returns chunks, and the answer step must refuse or say that the law does not answer them.

In [ ]:
def show(ids, gold):
    parts = [g.strip() for g in gold.split(";")]
    return " | ".join(("✓ " if any(ref_of[c].startswith(g) or g.startswith(ref_of[c]) for g in parts) else "") + ref_of[c] for c in ids[:5])

mine = ["B-01", "B-02", "B-03", "B-04", "B-05", "B-06", "B-S1", "B-S2", "B-S3"]
for q in mine:
    question, gold = questions[q]["question"], questions[q]["gold_source"]
    print(f"{q}: {question}\n  GOLD:   {gold}")
    for name, fn in [("dense", dense), ("BM25", keyword), ("hybrid", hybrid)]:
        print(f"  {name:7} {show(fn(question), gold)}")
    print()

**Comment:** _(write 1–2 sentences after running)_

## 8. A picture of the chunks
As in my earlier retrieval project, UMAP draws the 768 numbers of each chunk on a page. Chunks with a similar meaning should be close; each colour is one area of our provision list.

In [ ]:
import umap, matplotlib.pyplot as plt
xy = umap.UMAP(n_neighbors=10, random_state=42).fit_transform(vectors)
for area in sorted({c["area"] for c in chunks}):
    pts = xy[[c["area"] == area for c in chunks]]
    plt.scatter(pts[:, 0], pts[:, 1], s=25, label=area)
plt.legend(fontsize=7); plt.title("105 chunks, coloured by area"); plt.axis("off"); plt.show()

**Comment:** _(write 1–2 sentences after running)_

## Next
- The measurement on all the team's `val` questions (Recall@5, MRR, the instructor's RAG metrics, W&B) is KAN-99; it extends `src/evaluation.ipynb`.